# AutoMLChain - Real Training Tests

Este notebook demonstra o workflow completo de fine-tuning com dataset real usando o Replicate API.

**⚠️ Aviso**: Este notebook faz chamadas reais à API do Replicate e incorre em custos.

**Custo estimado**: ~$0.10 - $0.50 para este teste completo.

**Dataset**: IMDB Movie Reviews (subset de 500 samples para teste)

## 1. Setup e Configuração

In [ ]:
# Setup - Instalar dependências
!pip install pydantic click httpx orjson structlog tenacity tqdm datasets huggingface_hub --quiet

import os
import sys

# Adicionar path do repositório clonado
sys.path.insert(0, '/content/automlchain')

# Alternativa: instalar do git se não tiver o código
try:
    from automlchain import AutoMLPipeline, Dataset
except ModuleNotFoundError:
    print("Instalando automlchain do git...")
    !pip install git+https://github.com/gumeeee/automlchain.git --quiet
    from automlchain import AutoMLPipeline, Dataset

print("✅ Dependências instaladas!")

In [ ]:
# Configurar API token do Replicate
# Opção 1: Cole seu token diretamente aqui
# REPLICATE_TOKEN = "r8_your_token_here"

# Opção 2: Usar secrets do Colab
from google.colab import userdata
REPLICATE_TOKEN = userdata.get('REPLICATE_API_TOKEN')

# Opção 3: Variável de ambiente
if not REPLICATE_TOKEN:
    REPLICATE_TOKEN = os.environ.get("REPLICATE_API_TOKEN", "")

if not REPLICATE_TOKEN:
    print("⚠️  AVISO: REPLICATE_API_TOKEN não está configurado!")
    print("\nOpções para configurar:")
    print("1. Cole seu token na célula acima: REPLICATE_TOKEN = 'r8_...'")
    print("2. Adicione secret no Colab: (🔑) > Adicionar segredo > REPLICATE_API_TOKEN")
    print("3. Configure via ambiente: !set REPLICATE_API_TOKEN=seu_token")
    print("\nObtenha seu token em: https://replicate.com/account/api-tokens")
else:
    print("✅ REPLICATE_API_TOKEN configurado")
    print(f"Token: {REPLICATE_TOKEN[:10]}...{REPLICATE_TOKEN[-4:]}")

In [ ]:
# Importar módulos do AutoMLChain
import warnings
warnings.filterwarnings('ignore')

from automlchain import (
    AutoMLPipeline,
    Dataset,
    EvaluationSuite,
    F1, Accuracy,
)
from automlchain.providers import MockProvider, ReplicateProvider, PricingProvider

print("✅ Módulos importados com sucesso")

## 2. Preparação do Dataset (HuggingFace)

In [ ]:
# Limpar TODOS os caches do HuggingFace
import shutil
import os

# Limpar cache principal
hf_cache = os.path.expanduser("~/.cache/huggingface")
if os.path.exists(hf_cache):
    print("🧹 Limpando ~/.cache/huggingface...")
    shutil.rmtree(hf_cache, ignore_errors=True)

# Limpar cache local do Colab
colab_cache = "/root/.cache/huggingface"
if os.path.exists(colab_cache):
    print("🧹 Limpando /root/.cache/huggingface...")
    shutil.rmtree(colab_cache, ignore_errors=True)

# Limpar datasets
datasets_cache = os.path.expanduser("~/.cache/datasets")
if os.path.exists(datasets_cache):
    print("🧹 Limpando ~/.cache/datasets...")
    shutil.rmtree(datasets_cache, ignore_errors=True)

print("✅ Todos os caches limpos")


In [ ]:
from datasets import load_dataset, DownloadMode

# Forçar download sem usar cache (evita problemas de cache corrompido)
print("📥 Baixando dataset IMDB do HuggingFace...")

try:
    dataset = load_dataset(
        "imdb",
        download_mode=DownloadMode.FORCE_REDOWNLOAD
    )
except Exception as e:
    print(f"Erro no download: {e}")
    print("Tentando novamente com cache limpo...")
    import shutil
    import os
    for cache_dir in [os.path.expanduser("~/.cache/huggingface"), 
                      os.path.expanduser("~/.cache/datasets"),
                      "/root/.cache/huggingface"]:
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir, ignore_errors=True)
    dataset = load_dataset("imdb", download_mode=DownloadMode.FORCE_REDOWNLOAD)



In [ ]:
# Visualizar exemplos
print("\n📝 Exemplos do dataset:")
for i in range(3):
    sample = dataset['train'][i]
    label = "positive" if sample['label'] == 1 else "negative"
    text = sample['text'][:150] + "..."
    print(f"\n[{i+1}] Label: {label}")
    print(f"    Text: {text}")

In [ ]:
# Criar subset para teste (500 samples para reducir custo)
# Usar stratified sampling para manter balanceamento de classes
# Separar por classe
positive_indices = [i for i, x in enumerate(dataset['train']) if x['label'] == 1]
negative_indices = [i for i, x in enumerate(dataset['train']) if x['label'] == 0]

# Selecionar 250 de cada classe
n_samples_per_class = 250
selected_indices = positive_indices[:n_samples_per_class] + negative_indices[:n_samples_per_class]

# Criar subset
subset = dataset['train'].select(selected_indices)

print(f"✅ Subset criado com {len(subset)} samples")
print(f"   - Positive: {sum(1 for x in subset if x['label'] == 1)}")
print(f"   - Negative: {sum(1 for x in subset if x['label'] == 0)}")

## 3. Conversão do Dataset

In [ ]:
# Converter para formato JSONL esperado pelo AutoMLChain
import json

def convert_to_automl_format(sample):
    """Converte sample do HF para formato AutoMLChain."""
    label = "positive" if sample['label'] == 1 else "negative"
    # Truncar texto para reduzir tamanho
    text = sample['text'][:500] if len(sample['text']) > 500 else sample['text']
    return {
        "input": text,
        "output": label,
        "label": sample['label']
    }

# Converter todos os samples
automl_data = [convert_to_automl_format(sample) for sample in subset]

# Criar dataset do AutoMLChain
train_dataset = Dataset(data=automl_data[:400], format="jsonl")  # 400 para treino
test_dataset = Dataset(data=automl_data[400:], format="jsonl")    # 100 para teste

print(f"✅ Dataset convertido:")
print(f"   - Train: {len(train_dataset)} samples")
print(f"   - Test: {len(test_dataset)} samples")

print(f"\n📝 Exemplo formatado:")
print(json.dumps(automl_data[0], indent=2))

In [ ]:
# Salvar dataset em arquivo para upload
train_file = "train_sentiment.jsonl"
test_file = "test_sentiment.jsonl"

with open(train_file, 'w') as f:
    for item in automl_data[:400]:
        f.write(json.dumps(item) + '\n')

with open(test_file, 'w') as f:
    for item in automl_data[400:]:
        f.write(json.dumps(item) + '\n')

train_size = os.path.getsize(train_file) / 1024  # KB
test_size = os.path.getsize(test_file) / 1024

print(f"✅ Arquivos salvos:")
print(f"   - {train_file}: {train_size:.1f} KB ({len(train_dataset)} samples)")
print(f"   - {test_file}: {test_size:.1f} KB ({len(test_dataset)} samples)")

## 4. Análise de Custos

In [ ]:
# Estimar custos antes de iniciar
pricing = PricingProvider("replicate")

# Modelo a ser usado
MODEL = "meta/llama-3-8b-instruct"
EPOCHS = 1  # Apenas 1 epoch para teste
BATCH_SIZE = 4

# Estimar tokens no dataset (aproximado: ~100 tokens por sample)
estimated_tokens_per_sample = 100
total_train_tokens = len(train_dataset) * estimated_tokens_per_sample

# Calcular custos
training_cost = pricing.estimate_training_cost(
    model=MODEL,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    dataset_size_tokens=total_train_tokens
)

inference_cost_per_sample = pricing.estimate_inference_cost(
    model=MODEL,
    num_tokens=50  # ~50 tokens por predição
)
total_inference_cost = inference_cost_per_sample * len(test_dataset)

print("💰 Estimativa de Custos")
print("=" * 50)
print(f"Modelo: {MODEL}")
print(f"Epochs: {EPOCHS}")
print(f"Samples de treino: {len(train_dataset)}")
print(f"Samples de teste: {len(test_dataset)}")
print("-" * 50)
print(f"Custo estimado de training: ${training_cost:.4f}")
print(f"Custo estimado de inference: ${total_inference_cost:.4f}")
print(f"Custo TOTAL estimado: ${training_cost + total_inference_cost:.4f}")
print("=" * 50)

## 5. Upload e Validação

In [ ]:
# Escolher provider
USE_MOCK = not REPLICATE_TOKEN

if USE_MOCK:
    print("🔧 Usando MockProvider (sem custos)")
    provider = MockProvider(api_key="mock", simulate_duration=2.0)
else:
    print("🚀 Usando Replicate Provider (custos reais)")
    provider = ReplicateProvider(api_key=REPLICATE_TOKEN)

# Criar pipeline
pipeline = AutoMLPipeline(provider=provider)

# Upload dataset
print("\n📤 Fazendo upload do dataset...")
pipeline.upload_dataset(train_file)
print(f"✅ Dataset de treino carregado: {train_file}")

In [ ]:
# Validar dataset
from automlchain.datasets.validators import SchemaValidator, DatasetValidator

validator = DatasetValidator()

# Validar uma amostra
sample_data = [{"input": item["input"], "output": item["output"]} for item in automl_data[:10]]
result = validator.validate(sample_data)

print("✅ Validação do Dataset")
print("=" * 40)
print(f"Válido: {result.is_valid}")
if not result.is_valid:
    print(f"Erros: {len(result.errors)}")
    for error in result.errors[:3]:
        print(f"  - {error}")
else:
    print("Schema OK, encoding OK, sem valores missing")

## 6. Treinamento

In [ ]:
# Configurar hyperparameters
from automlchain.core.config import HyperParams

params = HyperParams(
    learning_rate=1e-4,
    lora_rank=16,
    batch_size=4,
    epochs=EPOCHS,
)

print("⚙️  Configuração de Hyperparameters:")
print(f"   Learning rate: {params.learning_rate}")
print(f"   LoRA rank: {params.lora_rank}")
print(f"   Batch size: {params.batch_size}")
print(f"   Epochs: {params.epochs}")

In [ ]:
# Iniciar treinamento
import time

print("🚀 Iniciando treinamento...")
print(f"   Modelo: {MODEL}")
print(f"   Dataset: {train_file}")

job = pipeline.train(
    model=MODEL,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=params.learning_rate,
)

print(f"\n✅ Job criado!")
print(f"   Job ID: {job.job_id}")
print(f"   Status: {job.status}")
print(f"   Modelo: {job.model}")

## 7. Monitoramento

In [ ]:
# Monitorar progresso do job
print("📊 Monitorando progresso...")
print("Pressione Kernel > Interrupt para cancelar")
print()

start_time = time.time()
last_status = ""

while True:
    status = provider.get_job_status(job.job_id)
    
    # Mostrar apenas se mudou
    if status.status != last_status:
        print(f"\n[{time.time() - start_time:.0f}s] Status: {status.status}")
        last_status = status.status
    
    # Mostrar progresso
    if hasattr(status, 'progress'):
        print(f"\r[{time.time() - start_time:.0f}s] Progress: {status.progress:.1f}%", end="")
    
    # Verificar se terminou
    if status.status in ["completed", "failed", "cancelled"]:
        print(f"\n\n✅ Job {status.status}!")
        if hasattr(status, 'metrics') and status.metrics:
            print(f"   Métricas: {status.metrics}")
        break
    
    time.sleep(5 if USE_MOCK else 30)  # Poll mais frequente para mock

total_time = time.time() - start_time
print(f"\n⏱️  Tempo total: {total_time:.1f} segundos")

## 8. Avaliação

In [ ]:
# Simular predictions (em produção, usar o modelo real)
# Aqui simulamos com predições baseadas em keywords
def simulate_prediction(text):
    """Simula predição baseada em keywords."""
    positive_keywords = ['great', 'excellent', 'amazing', 'love', 'best', 'fantastic', 'wonderful', 'good']
    negative_keywords = ['terrible', 'bad', 'worst', 'awful', 'hate', 'boring', 'horrible', 'poor']
    
    text_lower = text.lower()
    pos_count = sum(1 for kw in positive_keywords if kw in text_lower)
    neg_count = sum(1 for kw in negative_keywords if kw in text_lower)
    
    if pos_count > neg_count:
        return "positive"
    elif neg_count > pos_count:
        return "negative"
    else:
        return "positive" if hash(text) % 2 == 0 else "negative"

# Gerar predictions para test set
predictions = [simulate_prediction(item["input"]) for item in automl_data[400:]]
references = [item["output"] for item in automl_data[400:]]

print(f"📊 Predictions geradas: {len(predictions)}")
print(f"📊 References: {len(references)}")

In [ ]:
# Avaliar com AutoMLChain
suite = EvaluationSuite()
suite.add_metric("f1", F1())
suite.add_metric("accuracy", Accuracy())

result = suite.evaluate(predictions, references)

print("\n📈 Resultados da Avaliação")
print("=" * 50)
print(result)

# Mostrar detalhes
for metric in result.metrics:
    indicator = "↑" if metric.higher_is_better else "↓"
    print(f"{metric.name}: {metric.value:.4f} {indicator}")

In [ ]:
# Análise de erros
from collections import Counter

errors = [p for p, r in zip(predictions, references) if p != r]
error_types = Counter(errors)

print("\n🔍 Análise de Erros")
print("=" * 50)
print(f"Total de errors: {len(errors)} / {len(predictions)}")
print(f"Taxa de erro: {len(errors)/len(predictions)*100:.1f}%")
print(f"\nDistribuição de erros:")
for error_type, count in error_types.items():
    print(f"  - Predito '{error_type}': {count}")

## 9. Custo Real

In [ ]:
# Calcular custo real baseado no pricing
final_cost = pricing.estimate_training_cost(
    model=MODEL,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    dataset_size_tokens=len(train_dataset) * 100
)

print("💰 Resumo de Custos")
print("=" * 50)
print(f"Estimado antes: ${training_cost + total_inference_cost:.4f}")
print(f"Real (estimado): ${final_cost:.4f}")
print("=" * 50)
print("\n📝 Nota: Para custos reais, verifique o dashboard do Replicate.")

## 10. Deploy e Inference (Opcional)

In [ ]:
# Se o training completou com sucesso, podemos fazer deploy
status = provider.get_job_status(job.job_id)

if status.status == "completed":
    print("🚀 Fazendo deploy do modelo...")
    try:
        deployed = pipeline.deploy()
        print(f"✅ Modelo deployado!")
        print(f"   Endpoint: {deployed.endpoint}")
        
        # Testar prediction
        test_text = "This movie was absolutely fantastic!"
        pred = deployed.predict(test_text)
        print(f"\n📝 Teste de prediction:")
        print(f"   Input: '{test_text}'")
        print(f"   Output: {pred}")
    except Exception as e:
        print(f"⚠️  Deploy não disponível: {e}")
        print("   (API do Replicate pode não suportar deploy direto)")
else:
    print(f"⏭️  Deploy pulado - job status: {status.status}")

## 11. Cleanup

In [ ]:
# Limpar arquivos temporários
import os

files_to_clean = [train_file, test_file]

print("🧹 Limpando arquivos temporários...")
for f in files_to_clean:
    if os.path.exists(f):
        os.remove(f)
        print(f"   ✓ Removido: {f}")

print("\n✅ Cleanup completo!")

---

## Resumo

Este notebook demonstrou:

1. **Setup** - Configuração de API keys e ambiente
2. **Dataset** - Download do HuggingFace e conversão para JSONL
3. **Validação** - Validação de schema e dados
4. **Custos** - Estimativa de custos antes do treinamento
5. **Treinamento** - Início de job de fine-tuning
6. **Monitoramento** - Acompanhamento de progresso
7. **Avaliação** - Métricas de classification
8. **Deploy** - Deploy do modelo treinado (se disponível)
9. **Cleanup** - Limpeza de recursos

### Para testes futuros:

- Aumente o número de samples para melhor qualidade
- Use mais epochs para melhor fine-tuning
- Experimente diferentes modelos (llama-3-70b, mistral-7b)
- Adicione augmentation de dados
- Implemente early stopping baseado em métricas